In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Muat Dataset
df_mentah = pd.read_csv('sample_data/case9_student_performance.csv')

# Tampilkan ringkasan awal
print("Shape Data:", df_mentah.shape)
print("\n--- Info Data ---")
df_mentah.info()
print("\n--- Deskripsi Numerik ---")
display(df_mentah.describe())

Shape Data: (800, 19)

--- Info Data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              800 non-null    object 
 1   usia                    778 non-null    float64
 2   jenis_kelamin           776 non-null    object 
 3   jenis_sekolah           779 non-null    object 
 4   pendidikan_orang_tua    773 non-null    object 
 5   pekerjaan_orang_tua     777 non-null    object 
 6   jam_belajar_per_minggu  773 non-null    float64
 7   persentase_kehadiran    774 non-null    float64
 8   ikut_ekskul             766 non-null    object 
 9   akses_internet          757 non-null    object 
 10  les_privat              776 non-null    object 
 11  waktu_tempuh_menit      770 non-null    float64
 12  nilai_matematika        768 non-null    float64
 13  nilai_bahasa_indonesia  769 non-null    float64
 14  n

,usia,jam_belajar_per_minggu,persentase_kehadiran,waktu_tempuh_menit,nilai_matematika,nilai_bahasa_indonesia,nilai_ipa,nilai_ips,nilai_bahasa_inggris,nilai_rata_rata
count,778.000000,773.000000,774.000000,770.000000,768.000000,769.000000,757.000000,769.000000,752.000000,775.000000
mean,14.840617,21.513583,71.814212,77.071429,71.006510,75.600780,68.597094,73.109233,67.128989,70.695484
std,2.046511,12.156850,15.866712,41.519906,19.902335,18.991515,20.438452,20.058730,22.223969,18.463492
min,12.000000,1.000000,45.000000,5.000000,18.000000,31.000000,13.000000,11.000000,0.000000,25.600000
25%,13.000000,11.000000,57.300000,40.000000,56.000000,61.000000,53.000000,59.000000,51.000000,55.900000
50%,15.000000,21.000000,71.650000,78.500000,72.000000,77.000000,70.000000,74.000000,70.000000,71.600000
75%,17.000000,32.000000,85.200000,113.750000,87.000000,94.000000,85.000000,90.000000,86.000000,86.400000
max,18.000000,42.000000,99.800000,150.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [9]:
# ---------------------------------------------------------
# TAHAP 2: AUDIT, PEMBERSIHAN DATA, & LAPORAN KUALITAS
# ---------------------------------------------------------

# 1. Definisikan Fungsi Cek Kualitas Data Per Grup
def cek_kualitas(d, kolom_num='nilai_rata_rata', kolom_grup='jenis_sekolah', kunci='student_id'):
    g  = d.groupby(kolom_grup, observed=True)[kolom_num]
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    outlier = (d[kolom_num] < q1 - 1.5 * iqr) | (d[kolom_num] > q3 + 1.5 * iqr)
    return pd.Series({
        'n_baris':    len(d),
        'n_duplikat': d.duplicated(subset=kunci).sum(),
        'n_missing':  d.isnull().sum().sum(),
        'n_outlier':  outlier.sum(),
    })

df_bersih = df_mentah.copy()

# 2. Hapus Duplikat berdasarkan student_id
df_bersih = df_bersih.drop_duplicates(subset=['student_id'])

# 3. Imputasi Kolom Kategorik Terlebih Dahulu (Agar jenis_sekolah tidak NaN)
kolom_kat = ['jenis_sekolah', 'jenis_kelamin', 'pendidikan_orang_tua',
             'pekerjaan_orang_tua', 'ikut_ekskul', 'akses_internet', 'les_privat']

for col in kolom_kat:
    if col in df_bersih.columns:
        modus_val = df_bersih[col].mode()[0]
        df_bersih[col] = df_bersih[col].fillna(modus_val)

# 4. Imputasi Kolom Nilai Akademik (Median per jenis_sekolah, fallback ke Median Global)
kolom_nilai = ['nilai_matematika', 'nilai_bahasa_indonesia', 'nilai_ipa', 'nilai_ips', 'nilai_bahasa_inggris']
for col in kolom_nilai:
    if col in df_bersih.columns:
        df_bersih[col] = df_bersih.groupby('jenis_sekolah')[col].transform(lambda x: x.fillna(x.median()))
        df_bersih[col] = df_bersih[col].fillna(df_bersih[col].median())

# Hitung ulang nilai_rata_rata agar akurat pasca-imputasi
df_bersih['nilai_rata_rata'] = df_bersih[kolom_nilai].mean(axis=1)

# 5. Imputasi Fitur Numerik Lainnya (usia, jam_belajar, kehadiran, waktu_tempuh)
kolom_num_lain = ['usia', 'jam_belajar_per_minggu', 'persentase_kehadiran', 'waktu_tempuh_menit']
for col in kolom_num_lain:
    if col in df_bersih.columns:
        df_bersih[col] = df_bersih.groupby('jenis_sekolah')[col].transform(lambda x: x.fillna(x.median()))
        df_bersih[col] = df_bersih[col].fillna(df_bersih[col].median())

# 6. Outlier Handling (Clipping IQR per jenis_sekolah)
for col in ['jam_belajar_per_minggu', 'persentase_kehadiran', 'waktu_tempuh_menit']:
    if col in df_bersih.columns:
        q1 = df_bersih.groupby('jenis_sekolah')[col].transform(lambda x: x.quantile(0.25))
        q3 = df_bersih.groupby('jenis_sekolah')[col].transform(lambda x: x.quantile(0.75))
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        df_bersih[col] = np.clip(df_bersih[col], lower_bound, upper_bound)

# 7. Buat Laporan Sebelum vs Sesudah Pembersihan & Simpan ke CSV
laporan = pd.DataFrame({
    'sebelum': cek_kualitas(df_mentah, 'nilai_rata_rata', 'jenis_sekolah', 'student_id'),
    'sesudah': cek_kualitas(df_bersih, 'nilai_rata_rata', 'jenis_sekolah', 'student_id'),
})

display(laporan)
laporan.to_csv('laporan_kualitas.csv')
print("Laporan kualitas berhasil dibuat dan disimpan ke laporan_kualitas.csv!")

,sebelum,sesudah
n_baris,800,800
n_duplikat,0,0
n_missing,511,0
n_outlier,0,0


Laporan kualitas berhasil dibuat dan disimpan ke laporan_kualitas.csv!


In [10]:
# 1. Groupby Lanjutan: Rata-rata nilai berdasarkan Pendidikan Orang Tua & Akses Internet
analisis_ortu_internet = df_bersih.groupby(
    ['pendidikan_orang_tua', 'akses_internet'],
    observed=True
)[['nilai_matematika', 'nilai_bahasa_inggris', 'nilai_rata_rata']].mean().reset_index()

print("--- Rata-rata Nilai Berdasarkan Pendidikan Orang Tua & Internet ---")
display(analisis_ortu_internet)

# 2. Reshape / Pivot Table: Rata-rata jam belajar per minggu berdasarkan Status Kelulusan & Les Privat
pivot_performa = df_bersih.pivot_table(
    index='les_privat',
    columns='status_kelulusan',
    values='jam_belajar_per_minggu',
    aggfunc=['mean', 'count']
)

print("\n--- Pivot Table: Jam Belajar vs Status Kelulusan & Les Privat ---")
display(pivot_performa)

--- Rata-rata Nilai Berdasarkan Pendidikan Orang Tua & Internet ---


,pendidikan_orang_tua,akses_internet,nilai_matematika,nilai_bahasa_inggris,nilai_rata_rata
0,S1,Tidak,65.750000,63.979167,67.808333
1,S1,Ya,68.570000,65.810000,69.715000
2,S2,Tidak,72.675000,68.250000,70.637500
3,S2,Ya,75.130841,69.588785,74.132710
4,SD,Tidak,70.975000,66.250000,69.437500
5,SD,Ya,70.655462,67.319328,71.468067
6,SMA,Tidak,70.926829,66.902439,71.029268
7,SMA,Ya,71.181818,68.481818,71.457273
8,SMP,Tidak,70.792453,65.981132,70.507547
9,SMP,Ya,71.612676,67.260563,71.485915



--- Pivot Table: Jam Belajar vs Status Kelulusan & Les Privat ---


mean             count            
status_kelulusan      Lulus Tidak Lulus Lulus Tidak Lulus
les_privat                                               
Tidak             30.022364   10.313636   313         220
Ya                26.316667    9.137931   180          87

In [11]:
# 1. Split Features (X) & Target (y)
X = df_bersih.drop(columns=['student_id', 'status_kelulusan'])
y = df_bersih['status_kelulusan']

# 2. Train-Test Split dengan Stratify (Klasifikasi Biner Imbalanced/Balanced)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026, stratify=y
)

# 3. Pengelompokan Kolom Fitur
num_features = [
    'usia', 'jam_belajar_per_minggu', 'persentase_kehadiran',
    'waktu_tempuh_menit', 'nilai_matematika', 'nilai_bahasa_indonesia',
    'nilai_ipa', 'nilai_ips', 'nilai_bahasa_inggris', 'nilai_rata_rata'
]

cat_features = [
    'jenis_kelamin', 'jenis_sekolah', 'pendidikan_orang_tua',
    'pekerjaan_orang_tua', 'ikut_ekskul', 'akses_internet', 'les_privat'
]

# 4. Pipeline ColumnTransformer
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# Fit hanya pada X_train, transform pada X_train dan X_test
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

In [12]:
import os

# Pastikan folder output sudah dibuat
os.makedirs('output', exist_ok=True)

# 1. Simpan Dataset Bersih
df_bersih.to_csv('output/data_bersih.csv', index=False)

# 2. Buat Kamus Data
kamus_data = pd.DataFrame({
    'nama_kolom': df_bersih.columns,
    'tipe_data': df_bersih.dtypes.values,
    'n_missing': df_bersih.isnull().sum().values,
    'n_unik': df_bersih.nunique().values
})
kamus_data.to_csv('output/kamus_data.csv', index=False)
display(kamus_data)

,nama_kolom,tipe_data,n_missing,n_unik
0,student_id,object,0,800
1,usia,float64,0,8
2,jenis_kelamin,object,0,2
3,jenis_sekolah,object,0,2
4,pendidikan_orang_tua,object,0,5
5,pekerjaan_orang_tua,object,0,6
6,jam_belajar_per_minggu,float64,0,42
7,persentase_kehadiran,float64,0,415
8,ikut_ekskul,object,0,2
9,akses_internet,object,0,2
